# NB02 Data Transformation
## Sector Rotation & Federal Reserve Rate Cycles
### DS105W Data for Data Science | Group Project 2025–2026

<div style="font-family:system-ui; padding:16px 24px; background:#FFFFFF;
     border-left:8px solid #ED9255; border-radius:8px;
     box-shadow:0 4px 12px rgba(0,0,0,0.08); max-width:640px; color:#212121">

**Team:** Git It Done

| Member | Role | Notebook |
|--------|------|----------|
| Hugo Whyte | Data Collection | NB01 |
| **Parthiv Chakadath** | Cleaning & Database | **This notebook** |
| Joel Saldanha | Analysis & Website | NB03 + GitHub Pages |

**Research Question:** *How do US equity sector returns vary across Federal Reserve rate hiking and cutting cycles, and which sectors consistently rotate into outperformance as monetary policy shifts?*


## Decisions Made Before Writing Any Code

Following the same workflow Hugo used in NB01, I document the cleaning and storage decisions upfront so the choices are transparent and reviewable. Decisions 1 to 5 and 7 were made before writing code. Decision 6 documents how I handled two data gaps that surfaced during validation in Section 7.

---

### 1 · Inputs and Outputs

| Direction | Path | Format |
|-----------|------|--------|
| Read | `data/raw/fred_*.json` (3 files) | FRED JSON |
| Read | `data/raw/av_*.json` (10 files) | Alpha Vantage JSON |
| Write | `data/processed/sector_rotation.db` | SQLite |

NB02 reads only from `data/raw/` and writes only to `data/processed/`. NB03 will read only from the SQLite database and never touches the raw JSON. This separation is the W03 save-before-transform rule extended one stage further.

---

### 2 · Database Schema

Two normalised tables linked by `date`, matching the schema agreed in our planning document.

| Table | Columns | Purpose |
|-------|---------|---------|
| `macro_indicators` | `date` (PK), `fed_funds`, `cpi`, `yield_spread`, `rate_cycle` | One row per month, shared across all sectors |
| `sector_returns` | `id` (PK), `date` (FK), `sector`, `monthly_return` | One row per sector per month |

Storing the data this way (rather than as a single wide merged frame) means the macro context for any month is stored once rather than repeated ten times across the ten sectors. This is the relational normalisation principle implied by the W10 lecture's discussion of `tconst` as a shared key between `title_basics` and `title_ratings`.

---

### 3 · Rate Cycle Labelling Rule

The team agreed (group chat, 13/04) on a hybrid rolling-mean approach rather than month-to-month diff or hardcoded FOMC announcement dates:

- Compute `fed_funds.diff().rolling(3).mean()`, the rolling 3-month mean of monthly changes
- `hiking` if positive, `cutting` if negative, `neutral` if zero or undefined
- Implemented with `np.select()` for vectorised labelling

Rationale: month-to-month diff alone is too noisy (a single flat month mid-cycle would flip the label); FOMC announcement dates would introduce a third data dependency we have not collected. The rolling mean is fully vectorised, uses only permitted pandas methods, and is data-driven from our own series. **Known limitation (Issue #9):** Joel identified during NB03 plotting that flat-rate periods such as 2010–2015 and 2021–2022 produce noise-driven label flips because the pure sign classification has no magnitude threshold. The aggregated per-cycle averages in NB03 are not materially affected (each regime averages over 100+ months), but the issue is left open as a candidate future improvement (e.g. minimum-magnitude threshold, wider rolling window, or NBER cross-check).

---

### 4 · Date Convention Reconciliation

FRED reports monthly observations as the **first day** of the month (`2024-01-01`). Alpha Vantage reports them as the **last trading day** of the month (`2024-01-31` or similar). A naïve merge on raw date matches nothing.

I convert both sides to a monthly period (`YYYY-MM`) before merging, which ignores the day entirely. This preserves month-level granularity and avoids fragile day-of-month assumptions.

---

### 5 · Foreign Key Integrity

SQLite does not enforce foreign keys by default. I protect referential integrity in two ways: (a) drop any month from `macro_indicators` where `fed_funds` is null, then filter `sector_returns` to keep only rows whose `date` exists in the macro table; and (b) enable `PRAGMA foreign_keys = ON` at the connection level so the database itself rejects any orphan rows on insert. The pandas filter prevents us from *trying* to write orphans; the PRAGMA prevents the database from *accepting* them. Both together is the professional belt-and-braces pattern.

---

### 6 · Handling Known Data Gaps

Two months in the merged frame have null macro values:

- **October 2025**: `cpi` is null (BLS publication gap following the US federal shutdown that month). `fed_funds` and `yield_spread` are intact, so this row is retained.
- **April 2026**: all three series are null (the most recent month, not yet fully published by FRED). This row is dropped via the FK integrity step in Decision 5.

Both are real-world publication issues, not code errors. NB03 should use null filters when analyses involve CPI specifically.

---

### 7 · Vectorisation Policy

All transformations use vectorised pandas/numpy operations. The single non-vectorised loop is the file-reading loop in Section 2, which is unavoidable since each iteration is a file-system call rather than a row-level operation.

## Section 1: Imports

In [16]:
import os
import json
import sqlite3
import numpy as np
import pandas as pd

print("✅Libraries loaded successfully.")

✅Libraries loaded successfully.


## Section 2: Locate and Load Raw JSON Files

Notebooks run from `notebooks/`, so I navigate one level up to reach `data/raw/`, using the same `os.path` pattern Hugo used in NB01. I build explicit file lists for the FRED series and the ETFs rather than globbing the directory - if a file is missing, I see it immediately rather than silently producing a partial dataset

In [17]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DIR = os.path.join(REPO_ROOT, "data", "raw")
PROCESSED_DIR = os.path.join(REPO_ROOT, "data", "processed")

os.makedirs(PROCESSED_DIR, exist_ok=True)

FRED_FILES = {
    "fred_fedfunds.json": "fed_funds",
    "fred_cpiaucsl.json": "cpi",
    "fred_t10y2y.json":   "yield_spread",
}

ETF_FILES = {
    "av_XLK.json":  "Technology",
    "av_XLF.json":  "Financials",
    "av_XLE.json":  "Energy",
    "av_XLV.json":  "Health Care",
    "av_XLU.json":  "Utilities",
    "av_XLY.json":  "Consumer Discretionary",
    "av_XLP.json":  "Consumer Staples",
    "av_XLI.json":  "Industrials",
    "av_XLB.json":  "Materials",
    "av_XLRE.json": "Real Estate",
}

expected = list(FRED_FILES.keys()) + list(ETF_FILES.keys())
missing = [f for f in expected if not os.path.exists(os.path.join(RAW_DIR, f))]

if missing:
    print(f"⚠️ {len(missing)} missing files — rerun NB01 before continuing:")
    for f in missing:
        print(f"  {f}")
else:
    print(f"✅ All {len(expected)} expected raw JSON files found in data/raw/")

✅ All 13 expected raw JSON files found in data/raw/


## Section 3: Flatten FRED Macro Data with `json_normalize`

I reload one raw FRED file independently to confirm its structure rather than relying on assumptions carried over from NB01.

In [18]:
with open(os.path.join(RAW_DIR, "fred_fedfunds.json")) as f:
    example = json.load(f)

print("Top-level keys:", list(example.keys()))
print("First observation:", example["observations"][0])
print("Fields per observation:", list(example["observations"][0].keys()))

Top-level keys: ['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations']
First observation: {'realtime_start': '2026-04-20', 'realtime_end': '2026-04-20', 'date': '1993-01-01', 'value': '3.02'}
Fields per observation: ['realtime_start', 'realtime_end', 'date', 'value']


Each FRED file has an `observations` array with `date` and `value` fields. I use `pd.json_normalize()` with `record_path="observations"` to flatten each into a tidy two-column frame, then merge the three series on `date`.

`value` is a string (as Hugo flagged in NB01). I convert with `pd.to_numeric(errors="coerce")` so FRED's occasional `"."` missing-value placeholder becomes `NaN` rather than raising. I use a null check after each load to catch missing values early in line with W05 methodology.

In [19]:
fred_frames = {}

for filename, column_name in FRED_FILES.items():
    with open(os.path.join(RAW_DIR, filename)) as f:
        data = json.load(f)

    df = pd.json_normalize(data, record_path="observations")
    df = df[["date", "value"]].copy()
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.rename(columns={"value": column_name})

    n_null = df[column_name].isnull().sum()
    print(f"{column_name:<14} {len(df):>4} rows   nulls: {n_null}   {df['date'].min().date()} → {df['date'].max().date()}")

    fred_frames[column_name] = df

df_macro = fred_frames["fed_funds"] \
    .merge(fred_frames["cpi"], on="date", how="outer") \
    .merge(fred_frames["yield_spread"], on="date", how="outer") \
    .sort_values("date") \
    .reset_index(drop=True)

print(f"\n✅ Combined macro frame: {df_macro.shape[0]} rows, {df_macro.shape[1]} columns")
df_macro.head()

fed_funds       399 rows   nulls: 0   1993-01-01 → 2026-03-01
cpi             399 rows   nulls: 1   1993-01-01 → 2026-03-01
yield_spread    400 rows   nulls: 1   1993-01-01 → 2026-04-01

✅ Combined macro frame: 400 rows, 4 columns


,date,fed_funds,cpi,yield_spread
0,1993-01-01,3.02,142.8,2.21
1,1993-02-01,3.03,143.1,2.16
2,1993-03-01,3.07,143.3,2.03
3,1993-04-01,2.96,143.8,2.13
4,1993-05-01,3.00,144.2,2.06


## Section 4: Label Rate Cycle Periods with `np.select()`

Applying the labelling rule from Decision 3. `np.select()` takes a list of conditions and a list of corresponding values, applies them all at once, and returns a default for rows that match no condition. This is faster and more readable than chained `if/elif` inside `.apply()`, and it is the exact pattern shown in our planning diagram.

In [20]:
df_macro["fed_funds_roll3"] = df_macro["fed_funds"].diff().rolling(3).mean()

conditions = [
    df_macro["fed_funds_roll3"] > 0,
    df_macro["fed_funds_roll3"] < 0,
]
choices = ["hiking", "cutting"]

df_macro["rate_cycle"] = np.select(conditions, choices, default="neutral")

print("Rate cycle distribution across full sample:")
print(df_macro["rate_cycle"].value_counts())
print()
print("Sample rows showing the labelling in action:")
df_macro[["date", "fed_funds", "fed_funds_roll3", "rate_cycle"]].iloc[[0, 3, 50, 150, 300, -1]]

Rate cycle distribution across full sample:
rate_cycle
hiking     194
cutting    161
neutral     45
Name: count, dtype: int64

Sample rows showing the labelling in action:


,date,fed_funds,fed_funds_roll3,rate_cycle
0,1993-01-01,3.02,NaN,neutral
3,1993-04-01,2.96,-0.020000,cutting
50,1997-03-01,5.39,0.033333,hiking
150,2005-07-01,3.26,0.156667,hiking
300,2018-01-01,1.41,0.086667,hiking
399,2026-04-01,NaN,NaN,neutral


## Section 5: Flatten Alpha Vantage ETF Data and Compute Monthly Returns

Alpha Vantage's `TIME_SERIES_MONTHLY_ADJUSTED` endpoint returns a nested dictionary keyed by date, each value a dict of OHLC + adjusted close + dividend fields. I read it with `pd.DataFrame.from_dict(orient="index")` and use the `5. adjusted close` field, since adjusted prices account for dividends and splits, giving the true total return of holding each ETF (Hugo's NB01 Step 6 explains this in detail).

Monthly returns use `.groupby("sector")["adjusted_close"].pct_change()`. The groupby is essential, because without it the return for January for one sector would be calculated against the prior sector's final month, which is nonsense. Each sector's first month therefore has a `NaN` return, which I drop before writing.

In [21]:
sector_frames = []

for filename, sector_name in ETF_FILES.items():
    with open(os.path.join(RAW_DIR, filename)) as f:
        data = json.load(f)

    ts = data["Monthly Adjusted Time Series"]

    df = pd.DataFrame.from_dict(ts, orient="index")
    df.index = pd.to_datetime(df.index)
    df = df.rename_axis("date").reset_index()

    df["adjusted_close"] = pd.to_numeric(df["5. adjusted close"], errors="coerce")
    df["sector"] = sector_name
    df = df[["date", "sector", "adjusted_close"]].sort_values("date").reset_index(drop=True)

    n_null = df["adjusted_close"].isnull().sum()
    print(f"{sector_name:<25} {len(df):>4} months   nulls: {n_null}   {df['date'].min().date()} → {df['date'].max().date()}")

    sector_frames.append(df)

df_sectors = pd.concat(sector_frames, ignore_index=True)

df_sectors["monthly_return"] = (
    df_sectors
    .groupby("sector")["adjusted_close"]
    .pct_change()
)

before = len(df_sectors)
df_sectors = df_sectors.dropna(subset=["monthly_return"]).reset_index(drop=True)
print(f"\n✅ Computed monthly returns. Dropped {before - len(df_sectors)} rows with NaN return (first month per sector).")
print(f"Final sector frame: {df_sectors.shape[0]} rows, {df_sectors['sector'].nunique()} sectors")
df_sectors.head()

Technology                 317 months   nulls: 0   1999-12-31 → 2026-04-17
Financials                 317 months   nulls: 0   1999-12-31 → 2026-04-17
Energy                     317 months   nulls: 0   1999-12-31 → 2026-04-17
Health Care                317 months   nulls: 0   1999-12-31 → 2026-04-17
Utilities                  317 months   nulls: 0   1999-12-31 → 2026-04-17
Consumer Discretionary     317 months   nulls: 0   1999-12-31 → 2026-04-17
Consumer Staples           317 months   nulls: 0   1999-12-31 → 2026-04-17
Industrials                317 months   nulls: 0   1999-12-31 → 2026-04-17
Materials                  317 months   nulls: 0   1999-12-31 → 2026-04-17
Real Estate                126 months   nulls: 0   2015-11-30 → 2026-04-17

✅ Computed monthly returns. Dropped 10 rows with NaN return (first month per sector).
Final sector frame: 2969 rows, 10 sectors


,date,sector,adjusted_close,monthly_return
0,2000-01-31,Technology,18.7338,-0.061621
1,2000-02-29,Technology,20.7050,0.105222
2,2000-03-31,Technology,22.4391,0.083753
3,2000-04-28,Technology,20.3790,-0.091808
4,2000-05-31,Technology,18.2595,-0.104004


## Section 6: Reconcile Date Conventions and Merge Macro + Sector Data

Applying the date convention reconciliation from Decision 4. Both frames get a `merge_date` helper column converted to monthly period, the merge happens on that, then the helper is renamed back to `date`. The `rate_cycle` label flows from the macro frame onto each sector row via the merge.

I use an inner join because a sector return without matching macro context is not useful for our research question, and vice versa. The inner join also handles XLRE's October 2015 start naturally, since months before that date simply don't have an XLRE row.

In [22]:
df_macro["merge_date"] = df_macro["date"].dt.to_period("M").dt.to_timestamp()
df_sectors["merge_date"] = df_sectors["date"].dt.to_period("M").dt.to_timestamp()

df_merged = pd.merge(
    df_sectors[["merge_date", "sector", "monthly_return"]],
    df_macro[["merge_date", "fed_funds", "cpi", "yield_spread", "rate_cycle"]],
    on="merge_date",
    how="inner",
)

df_merged = df_merged.rename(columns={"merge_date": "date"})

print(f"✅ Merged frame: {df_merged.shape[0]} rows, {df_merged.shape[1]} columns")
print(f"Date range: {df_merged['date'].min().date()} → {df_merged['date'].max().date()}")
print(f"Rate cycle distribution in merged data:")
print(df_merged["rate_cycle"].value_counts())
print(f"\nRows per sector (XLRE will be lower, as expected):")
print(df_merged["sector"].value_counts().sort_index())

✅ Merged frame: 2969 rows, 7 columns
Date range: 2000-01-01 → 2026-04-01
Rate cycle distribution in merged data:
rate_cycle
hiking     1406
cutting    1160
neutral     403
Name: count, dtype: int64

Rows per sector (XLRE will be lower, as expected):
sector
Consumer Discretionary    316
Consumer Staples          316
Energy                    316
Financials                316
Health Care               316
Industrials               316
Materials                 316
Real Estate               125
Technology                316
Utilities                 316
Name: count, dtype: int64


## Section 7: Validation Checks

Before writing to the database I check for nulls in key columns, impossible returns (>100% in a single month is almost certainly a data error), and cycle-label coverage.

In [23]:
print("Missing values in merged frame:")
nulls = df_merged.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "  ✅ None")
print()

extreme = df_merged[(df_merged["monthly_return"] > 1.0) | (df_merged["monthly_return"] < -0.5)]
print(f"Extreme monthly returns (>100% or <-50%): {len(extreme)}")
if len(extreme) > 0:
    print(extreme)
print()

cycle_coverage = df_merged.groupby("rate_cycle")["date"].agg(["count", "min", "max"])
print("Rate cycle coverage:")
print(cycle_coverage)
print()

sector_cycle = df_merged.groupby(["sector", "rate_cycle"]).size().unstack(fill_value=0)
print("Rows per sector × cycle (for NB03's groupby analysis):")
sector_cycle

Missing values in merged frame:
fed_funds       10
cpi             20
yield_spread    10
dtype: int64

Extreme monthly returns (>100% or <-50%): 0

Rate cycle coverage:
            count        min        max
rate_cycle                             
cutting      1160 2000-09-01 2026-03-01
hiking       1406 2000-01-01 2023-10-01
neutral       403 2002-09-01 2026-04-01

Rows per sector × cycle (for NB03's groupby analysis):


rate_cycle,cutting,hiking,neutral
sector,,,
Consumer Discretionary,125,149,42
Consumer Staples,125,149,42
Energy,125,149,42
Financials,125,149,42
Health Care,125,149,42
Industrials,125,149,42
Materials,125,149,42
Real Estate,35,65,25
Technology,125,149,42


The next two cells trace how I identified the specific months involved. The first finds any month with at least one null macro indicator. The second inspects those months in `df_macro` to see which series specifically is missing. This is what surfaced the October 2025 CPI gap and the April 2026 publication-lag issue documented in Decision 6.

In [24]:
null_months = df_merged[df_merged[["fed_funds", "cpi", "yield_spread"]].isnull().any(axis=1)]["date"].unique()
print(f"Months with any macro null: {sorted(null_months)}")

Months with any macro null: [Timestamp('2025-10-01 00:00:00'), Timestamp('2026-04-01 00:00:00')]


In [25]:
df_macro[df_macro["date"].isin(pd.to_datetime(["2025-10-01", "2026-04-01"]))]

,date,fed_funds,cpi,yield_spread,fed_funds_roll3,rate_cycle,merge_date
393,2025-10-01,4.09,NaN,0.54,-0.08,cutting,2025-10-01
399,2026-04-01,NaN,NaN,NaN,NaN,neutral,2026-04-01


## Section 8: Build the Two Final Tables for SQLite

Building the two tables defined in Decision 2, applying the foreign key integrity step from Decision 5: drop macro rows where `fed_funds` is null, then filter `sector_returns` to keep only dates that exist in the macro table.

In [26]:
macro_table = (
    df_macro[["date", "fed_funds", "cpi", "yield_spread", "rate_cycle"]]
    .dropna(subset=["fed_funds"])
    .drop_duplicates(subset="date")
    .sort_values("date")
    .reset_index(drop=True)
)
macro_table["date"] = macro_table["date"].dt.strftime("%Y-%m-%d")

valid_dates = set(macro_table["date"])

sector_table = (
    df_merged[["date", "sector", "monthly_return"]]
    .assign(date=lambda d: d["date"].dt.strftime("%Y-%m-%d"))
    .loc[lambda d: d["date"].isin(valid_dates)]
    .sort_values(["date", "sector"])
    .reset_index(drop=True)
)
sector_table.insert(0, "id", range(1, len(sector_table) + 1))

print(f"macro_indicators : {len(macro_table):>5} rows")
print(f"sector_returns   : {len(sector_table):>5} rows")
print("\nmacro_indicators preview:")
print(macro_table.head(3))
print("\nsector_returns preview:")
print(sector_table.head(3))

macro_indicators :   399 rows
sector_returns   :  2959 rows

macro_indicators preview:
         date  fed_funds    cpi  yield_spread rate_cycle
0  1993-01-01       3.02  142.8          2.21    neutral
1  1993-02-01       3.03  143.1          2.16    neutral
2  1993-03-01       3.07  143.3          2.03    neutral

sector_returns preview:
   id        date                  sector  monthly_return
0   1  2000-01-01  Consumer Discretionary       -0.124270
1   2  2000-01-01        Consumer Staples        0.008682
2   3  2000-01-01                  Energy        0.008125


## Section 9: Write to SQLite Database

The schema is defined explicitly with `CREATE TABLE` rather than letting pandas infer it. Explicit definitions let me declare primary and foreign keys, lock in the column types, and produce a self-documenting schema. I enable `PRAGMA foreign_keys = ON` at the connection level so the database itself rejects orphan inserts (Decision 5). The connection is opened with a `with` block so it commits and closes cleanly even if anything in the write step fails.

In [27]:
db_path = os.path.join(PROCESSED_DIR, "sector_rotation.db")

if os.path.exists(db_path):
    os.remove(db_path)

schema_sql = """
CREATE TABLE macro_indicators (
    date         TEXT PRIMARY KEY,
    fed_funds    REAL,
    cpi          REAL,
    yield_spread REAL,
    rate_cycle   TEXT NOT NULL
);

CREATE TABLE sector_returns (
    id             INTEGER PRIMARY KEY,
    date           TEXT NOT NULL,
    sector         TEXT NOT NULL,
    monthly_return REAL NOT NULL,
    FOREIGN KEY (date) REFERENCES macro_indicators(date)
);

CREATE INDEX idx_sector_returns_date   ON sector_returns(date);
CREATE INDEX idx_sector_returns_sector ON sector_returns(sector);
"""

with sqlite3.connect(db_path) as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executescript(schema_sql)
    macro_table.to_sql("macro_indicators", conn, if_exists="append", index=False)
    sector_table.to_sql("sector_returns", conn, if_exists="append", index=False)

print(f"✅ Database written to {db_path}")

✅ Database written to /files/group-project-git-it-done/data/processed/sector_rotation.db


## Section 10: Verify the Database by Reading It Back

Following the W10 pattern: reopen the connection, query the `sqlite_master` catalogue to confirm the tables exist, and inspect the schema of each with `PRAGMA table_info()`. This confirms the `CREATE TABLE` statements actually built the columns and types I specified.

In [28]:
with sqlite3.connect(db_path) as conn:
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn
    )
    print("Tables in database:")
    print(tables)
    print()

    print("Schema of macro_indicators:")
    print(pd.read_sql("PRAGMA table_info(macro_indicators);", conn))
    print()

    print("Schema of sector_returns:")
    print(pd.read_sql("PRAGMA table_info(sector_returns);", conn))

Tables in database:
               name
0  macro_indicators
1    sector_returns

Schema of macro_indicators:
   cid          name  type  notnull dflt_value  pk
0    0          date  TEXT        0       None   1
1    1     fed_funds  REAL        0       None   0
2    2           cpi  REAL        0       None   0
3    3  yield_spread  REAL        0       None   0
4    4    rate_cycle  TEXT        1       None   0

Schema of sector_returns:
   cid            name     type  notnull dflt_value  pk
0    0              id  INTEGER        0       None   1
1    1            date     TEXT        1       None   0
2    2          sector     TEXT        1       None   0
3    3  monthly_return     REAL        1       None   0


## Section 11: Demonstrate SQL Queries for NB03

Now that the database exists, I run a small set of queries against it to (a) verify the data is queryable as intended and (b) give Joel a working set of SQL patterns to lift directly into NB03's analysis. Each query mirrors a pattern from the W10 lecture.

**Query A**, `COUNT` + `WHERE`: how many months fell in a hiking cycle??
**Query B**, `GROUP BY` + `COUNT` + `ORDER BY`: how many months fall into each rate cycle?
**Query C**, `WHERE` + `IS NOT NULL`: how many months have complete macro data, excluding the known October 2025 CPI gap?
**Query D**, `JOIN` with table aliases: sector returns enriched with macro context, the core query NB03 will build on.

In [29]:
with sqlite3.connect(db_path) as conn:

# Query A: COUNT + WHERE — months in a hiking cycle
    print("Query A: Months in a hiking cycle:")
    hiking_count = pd.read_sql("""
        SELECT COUNT(*) AS n_hiking_months
        FROM macro_indicators
        WHERE rate_cycle = 'hiking';
    """, conn)
    print(hiking_count)
    print()

    # Query B — GROUP BY + COUNT + ORDER BY: rate cycle distribution
    print("Query B — Months per rate cycle:")
    cycle_counts = pd.read_sql("""
        SELECT rate_cycle, COUNT(*) AS n_months
        FROM macro_indicators
        GROUP BY rate_cycle
        ORDER BY n_months DESC;
    """, conn)
    print(cycle_counts)
    print()

    # Query C — WHERE + IS NOT NULL: complete macro data
    print("Query C — Months with complete macro data (CPI non-null):")
    complete = pd.read_sql("""
        SELECT COUNT(*) AS n_complete
        FROM macro_indicators
        WHERE cpi IS NOT NULL;
    """, conn)
    print(complete)
    print()

    # Query D — JOIN with aliases: sector returns + macro context
    print("Query D — Latest 5 Technology returns with macro context (JOIN):")
    sample_join = pd.read_sql("""
        SELECT s.date,
               s.sector,
               s.monthly_return,
               m.fed_funds,
               m.rate_cycle
        FROM sector_returns AS s
        JOIN macro_indicators AS m
          ON s.date = m.date
        WHERE s.sector = 'Technology'
        ORDER BY s.date DESC
        LIMIT 5;
    """, conn)
    print(sample_join)

Query A: Months in a hiking cycle:
   n_hiking_months
0              194

Query B — Months per rate cycle:
  rate_cycle  n_months
0     hiking       194
1    cutting       161
2    neutral        44

Query C — Months with complete macro data (CPI non-null):
   n_complete
0         398

Query D — Latest 5 Technology returns with macro context (JOIN):
         date      sector  monthly_return  fed_funds rate_cycle
0  2026-03-01  Technology       -0.041060       3.64    cutting
1  2026-02-01  Technology       -0.035585       3.64    cutting
2  2026-01-01  Technology       -0.000625       3.64    cutting
3  2025-12-01  Technology        0.007526       3.72    cutting
4  2025-11-01  Technology       -0.048091       3.88    cutting


## Summary

In this notebook I:

1. **Documented all decisions** before writing any code: inputs/outputs, schema, rate cycle rule, date reconciliation, FK integrity, known data gaps, and vectorisation policy
2. **Loaded 13 raw JSON files** from `data/raw/` saved by NB01, comprising 3 FRED macro series and 10 Alpha Vantage sector ETFs
3. **Flattened FRED data** with `pd.json_normalize()` (W07) and merged the three series on `date`
4. **Labelled rate cycles** using vectorised `np.select()` on the rolling 3-month mean of `fed_funds.diff()`
5. **Flattened Alpha Vantage data** with `pd.DataFrame.from_dict()` and computed monthly returns per sector with `.groupby().pct_change()`
6. **Reconciled date conventions** between FRED (start-of-month) and Alpha Vantage (end-of-month trading day) via monthly-period conversion
7. **Merged macro and sector data** on `date` with `pd.merge()` (W08), validated row counts, and inspected nulls
8. **Built two normalised tables** with foreign-key integrity, `macro_indicators` (399 rows) and `sector_returns` (2,959 rows), and wrote them to a SQLite database with `PRAGMA foreign_keys = ON`
9. **Verified the database** by querying `sqlite_master` and `PRAGMA table_info()` (W10)
10. **Demonstrated SQL query patterns for NB03** including `COUNT`, `GROUP BY + ORDER BY`, `WHERE … IS NOT NULL`, and `JOIN` with table aliases (W10)

---

**Next step, NB03 (Joel):** Read from `sector_rotation.db` using `sqlite3.connect()` and `pd.read_sql()`, compute mean returns per sector × cycle with `.groupby().agg()`, build a sector × cycle pivot table with `.pivot_table()`, and produce the three planned visualisations (bar chart, heatmap, line chart with shaded cycle periods). The four query patterns demonstrated in Section 11 should cover most of the SQL Joel needs.

## Handoff Notes

**Date format and merge convention:**

All `date` values in both tables are stored as strings in `YYYY-MM-DD` format, set to the first of the month (e.g. `2024-01-01`). This was necessary because FRED publishes monthly data on the first of the month while Alpha Vantage publishes on the last trading day, so the two sources were reconciled to a single first-of-month convention before merging (Decision 4). Joel should treat `date` as a month-level label rather than a literal trading day. For pandas operations, `pd.to_datetime(df["date"])` will convert it back to a datetime if needed.

**Known data gaps, heads-up for NB03:**

Two months in the merged frame have null macro values that NB03 will need to handle: October 2025 is missing CPI (a known publication gap following the US federal shutdown that month) and April 2026 was dropped entirely (most recent month, not yet fully published by FRED). Any analysis involving CPI specifically should use `WHERE cpi IS NOT NULL` (or `.dropna()` in pandas) on the macro table. The Section 11 query pattern shows the SQL form.

**XLRE small-sample warning for visualisations:**

XLRE (Real Estate) only has data from October 2015, so any sector × cycle cell with very few observations (e.g. XLRE during the 2015 to 2018 hiking cycle) should be flagged with a small-sample note in NB03's heatmap. The row counts per sector × cycle from Section 7 give Joel the exact numbers to apply this filter.